# 01 — UC2 Centralized Baseline (DSCL)

Train a centralized LSTM model (Data-Sharing Centralized Learning)
as the upper-bound baseline. One model trained on all APs' data.

Since the centralized model doesn't depend on α (it sees all data),
we only need to run it once per alpha partition (the test split differs).

In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import Counter


sys.path.insert(0, uc2.LIB_DIR)

## Configuration

In [ ]:
ALPHAS = [0.5, 1.0, 5.0, 10.0]
MODEL = "lstm"

# Override any defaults here
OVERRIDES = dict(
    num_glob_iters=100,
    local_epochs=50,
    early_stopping_patience=50,
    # device="cpu",  # uncomment if no GPU
)

## Train Centralized Model

The `Centralized` server aggregates ALL users' data into a single 
training set. We run it for each alpha because the data partition 
(and hence the test set) differs per alpha.

In [ ]:
results_centralized = {}

for alpha in ALPHAS:
    print(f"\n{'='*60}")
    print(f"  CENTRALIZED — α={alpha}")
    print(f"{'='*60}")
    
    try:
        server, result = uc2.run_experiment(
            algorithm="Centralized",
            alpha=alpha,
            **OVERRIDES
        )
        results_centralized[alpha] = result
        
        # Print final metrics
        glob_metrics = result["metrics"]["glob_test_metric"]
        if glob_metrics:
            best_idx = np.argmin([m.get("unscaled_mae", float("inf")) 
                                  for m in glob_metrics])
            best = glob_metrics[best_idx]
            print(f"\n  Best round: {best_idx}")
            print(f"  MAE (scaled):   {best.get('mae', 'N/A'):.4f}")
            print(f"  MAE (unscaled): {best.get('unscaled_mae', 'N/A'):.4f}")
            print(f"  MAPE:           {best.get('unscaled_mape', 'N/A'):.4f}")
            
    except Exception as e:
        print(f"  [ERROR] {e}")
        import traceback
        traceback.print_exc()

## Training Curves

In [ ]:
import matplotlib.pyplot as plt
uc2.setup_plot_style()

fig, ax = plt.subplots(figsize=(10, 5))
for alpha, r in sorted(results_centralized.items()):
    glob_metrics = r["metrics"].get("glob_test_metric", [])
    maes = [m.get("unscaled_mae") for m in glob_metrics]
    ax.plot(maes, label=f"α={alpha}", linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Unscaled MAE (MB)")
ax.set_title("Centralized Training Convergence")
ax.legend()
plt.tight_layout()
plt.show()

## Summary

Centralized results saved to `results/centralized/alpha_{α}/lstm/rep_0/`.
These serve as the upper-bound baseline for comparison with FL methods.